# Step 6 — Run the reference model

**Learning objective:** Run OGS against the prepared mesh and plot the newly generated potential field.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Run:** OGS only. Mesh generation belongs to earlier steps and will not be repeated here.

In [ ]:
(ROOT/'results').mkdir(exist_ok=True)
run('ogs','-o','results','models/excavation.prj')
run(sys.executable,'scripts/check_results.py')

**Visual checkpoint:** plot the solved potential from the last VTU.

In [ ]:
from matplotlib.tri import Triangulation
files=sorted((ROOT/'results').glob('excavation_diffusion_ts_1_*.vtu'))
assert len(files)==1, f'Expected one reference result, found {files}'
m=meshio.read(files[0]);tri=np.vstack([c.data for c in m.cells if c.type=='triangle'])
u=np.asarray(m.point_data['pressure']).reshape(-1)
fig,ax=plt.subplots(figsize=(12,5.5))
im=ax.tricontourf(Triangulation(m.points[:,0],m.points[:,1],tri),u,levels=np.linspace(30,60,31),cmap='coolwarm')
fig.colorbar(im,ax=ax,label='Potential [model units]')
ax.set(xlim=(0,65),ylim=(0,30),xlabel='x [m]',ylabel='y [m]',title='OGS reference solution (60 / 30)')
ax.set_aspect('equal');plt.show()

**Interpretation / PASS:** This is the first *computed solution*. The colours denote scalar `pressure` **in model units**; red corresponds to higher values and blue to lower values. `check_results.py` confirms that exported boundary-node values satisfy the prescribed Dirichlet conditions. Do not interpret the field as calibrated kPa. **Next:** rerun the same geometry with u = 50 on the left.